# 4.3 Subspaces of Vector Spaces

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4 — Vector Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply
from linalg_core.elimination import solve
from linalg_core.vecspace import is_in_span
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import random

random.seed(43)

print("Setup complete.")

---

## 2. Motivation

Not every subset of a vector space is itself a vector space. But some special
subsets show up **everywhere** in applications:

- The set of all solutions to $A\mathbf{x} = \mathbf{0}$ (the **null space**).
- The set of all linear combinations of a given set of vectors (the **span**).
- Lines and planes through the origin in $\mathbb{R}^2$ and $\mathbb{R}^3$.

What makes these sets special? They are **closed** under the two operations that
define a vector space: vector addition and scalar multiplication. A subset with
this property is called a **subspace**.

In this notebook we formalize the definition, build a testing function, catalog
the subspaces of $\mathbb{R}^2$ and $\mathbb{R}^3$, connect spans to subspaces,
and preview the null space $-$ the subspace that will dominate the rest of the course.

---

## 3. Build — Core Concepts

We develop the subspace concept in six steps, starting with the definition and
working toward the key examples.

### 3.1 The Subspace Test — Three Conditions

**Definition (Larson, Sec. 4.3).** A nonempty subset $W$ of a vector space $V$
is a **subspace** of $V$ if $W$ is itself a vector space under the same operations.

Rather than checking all ten vector-space axioms, we only need **three conditions**
(Theorem 4.5):

| # | Condition | Meaning |
|---|---|---|
| 1 | $\mathbf{0} \in W$ | $W$ contains the zero vector |
| 2 | $\mathbf{u} + \mathbf{v} \in W$ for all $\mathbf{u}, \mathbf{v} \in W$ | Closed under addition |
| 3 | $c\mathbf{u} \in W$ for all $\mathbf{u} \in W$, $c \in \mathbb{R}$ | Closed under scalar multiplication |

**Why only three?** The remaining axioms (commutativity, associativity, distributivity, etc.)
are inherited from the parent space $V$ because $W$ uses the *same* operations.

We implement a sampling-based test: given a finite collection of vectors that we claim
belong to $W$, we check the three conditions on all pairs and a range of scalars.

In [ ]:
def is_subspace_candidate(vectors, zero_vec, membership_test, scalars=None):
    """Test the 3 subspace conditions on a sample of vectors.

    Parameters
    ----------
    vectors : list of list
        Sample vectors believed to lie in the candidate subspace.
    zero_vec : list
        The zero vector of the ambient space.
    membership_test : callable
        A function f(v) -> bool that returns True if v belongs to the set.
    scalars : list of float or None
        Scalars to test closure under scalar multiplication.
        Defaults to [-2, -1, -0.5, 0, 0.5, 1, 2, 3].

    Returns
    -------
    (passes: bool, report: dict)
    """
    if scalars is None:
        scalars = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

    report = {"zero": False, "addition": True, "scalar_mul": True, "details": []}

    # Condition 1: contains zero
    report["zero"] = membership_test(zero_vec)
    if not report["zero"]:
        report["details"].append("FAIL: zero vector is not in the set.")

    # Condition 2: closed under addition
    for i, u in enumerate(vectors):
        for j, v in enumerate(vectors):
            s = [u[k] + v[k] for k in range(len(u))]
            if not membership_test(s):
                report["addition"] = False
                report["details"].append(
                    f"FAIL addition: v{i} + v{j} = {s} is not in the set."
                )

    # Condition 3: closed under scalar multiplication
    for i, u in enumerate(vectors):
        for c in scalars:
            s = [c * u[k] for k in range(len(u))]
            if not membership_test(s):
                report["scalar_mul"] = False
                report["details"].append(
                    f"FAIL scalar mul: {c} * v{i} = {s} is not in the set."
                )

    passes = report["zero"] and report["addition"] and report["scalar_mul"]
    return passes, report


print("is_subspace_candidate defined.")

### 3.2 Trivial Subspaces

Every vector space $V$ has two **trivial subspaces**:

1. $\{\mathbf{0}\}$ — the set containing only the zero vector.
2. $V$ itself.

**Why is $\{\mathbf{0}\}$ a subspace?**
- Contains $\mathbf{0}$. \checkmark
- $\mathbf{0} + \mathbf{0} = \mathbf{0} \in \{\mathbf{0}\}$. \checkmark
- $c \cdot \mathbf{0} = \mathbf{0} \in \{\mathbf{0}\}$ for all $c$. \checkmark

**Why is $V$ a subspace of itself?** It trivially satisfies all three conditions since
every vector in $V$ is already in $V$.

Any subspace other than $\{\mathbf{0}\}$ and $V$ is called a **proper subspace**.

In [ ]:
# Test {0} in R^3
zero_3 = [0.0, 0.0, 0.0]

def in_zero_set(v):
    return all(abs(x) < EPSILON for x in v)

ok, report = is_subspace_candidate([zero_3], zero_3, in_zero_set)
print(f"{{0}} is a subspace of R^3: {ok}")

# Test V = R^3 itself
def in_R3(v):
    return len(v) == 3

sample_vecs = [[1, 2, 3], [-1, 0, 5], [0.5, -3.7, 2.1]]
ok, report = is_subspace_candidate(sample_vecs, zero_3, in_R3)
print(f"R^3 is a subspace of R^3: {ok}")

### 3.3 Subspaces of $\mathbb{R}^2$

The **only** subspaces of $\mathbb{R}^2$ are:

| Subspace | Geometric description |
|---|---|
| $\{\mathbf{0}\}$ | The origin |
| Lines through the origin | $W = \{t\mathbf{v} : t \in \mathbb{R}\}$ for some nonzero $\mathbf{v}$ |
| $\mathbb{R}^2$ | The entire plane |

**Example 1:** The line $y = 2x$ is a subspace. Every point on this line has the
form $(t, 2t) = t(1, 2)$, which is a scalar multiple of $(1, 2)$.

**Example 2:** The line $y = 2x + 1$ is **not** a subspace. It does not pass through
the origin: $(0, 0)$ does not satisfy $0 = 2(0) + 1$.

In [ ]:
zero_2 = [0.0, 0.0]

# Line y = 2x  (subspace)
def on_line_through_origin(v):
    """Test if (x, y) satisfies y = 2x."""
    return abs(v[1] - 2 * v[0]) < EPSILON

sample_line = [[1, 2], [-3, -6], [0.5, 1.0]]
ok, report = is_subspace_candidate(sample_line, zero_2, on_line_through_origin)
print(f"y = 2x is a subspace of R^2: {ok}")
for d in report["details"]:
    print(f"  {d}")

print()

# Line y = 2x + 1  (NOT a subspace)
def on_shifted_line(v):
    """Test if (x, y) satisfies y = 2x + 1."""
    return abs(v[1] - 2 * v[0] - 1) < EPSILON

sample_shifted = [[0, 1], [1, 3], [-1, -1]]
ok, report = is_subspace_candidate(sample_shifted, zero_2, on_shifted_line)
print(f"y = 2x + 1 is a subspace of R^2: {ok}")
for d in report["details"]:
    print(f"  {d}")

### 3.4 Subspaces of $\mathbb{R}^3$

The **only** subspaces of $\mathbb{R}^3$ are:

| Subspace | Dimension | Geometric description |
|---|---|---|
| $\{\mathbf{0}\}$ | 0 | The origin |
| Lines through the origin | 1 | $\text{span}(\mathbf{v})$ for nonzero $\mathbf{v}$ |
| Planes through the origin | 2 | $\text{span}(\mathbf{v}_1, \mathbf{v}_2)$ for independent $\mathbf{v}_1, \mathbf{v}_2$ |
| $\mathbb{R}^3$ | 3 | The entire space |

**Key fact:** Every subspace of $\mathbb{R}^3$ can be described as the solution set
of a homogeneous system $A\mathbf{x} = \mathbf{0}$, or equivalently, as the span
of a set of vectors.

**Example:** The plane $x + y + z = 0$ is a subspace of $\mathbb{R}^3$.

In [ ]:
zero_3 = [0.0, 0.0, 0.0]

# Plane x + y + z = 0  (subspace)
def on_plane_through_origin(v):
    return abs(v[0] + v[1] + v[2]) < EPSILON

sample_plane = [[1, -1, 0], [0, 1, -1], [2, 1, -3]]
ok, report = is_subspace_candidate(sample_plane, zero_3, on_plane_through_origin)
print(f"x + y + z = 0 is a subspace of R^3: {ok}")

print()

# Line through origin: span of (1, 2, 3)
def on_line_R3(v):
    """Test if v is a scalar multiple of (1, 2, 3)."""
    if all(abs(x) < EPSILON for x in v):
        return True
    for i in range(3):
        if abs([1, 2, 3][i]) > EPSILON:
            t = v[i] / [1, 2, 3][i]
            return all(abs(v[j] - t * [1, 2, 3][j]) < EPSILON for j in range(3))
    return False

sample_line_3d = [[1, 2, 3], [-2, -4, -6], [0.5, 1.0, 1.5]]
ok, report = is_subspace_candidate(sample_line_3d, zero_3, on_line_R3)
print(f"span((1,2,3)) is a subspace of R^3: {ok}")

print()

# Plane x + y + z = 1  (NOT a subspace)
def on_shifted_plane(v):
    return abs(v[0] + v[1] + v[2] - 1) < EPSILON

sample_shifted_plane = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
ok, report = is_subspace_candidate(sample_shifted_plane, zero_3, on_shifted_plane)
print(f"x + y + z = 1 is a subspace of R^3: {ok}")
for d in report["details"]:
    print(f"  {d}")

### 3.5 Span Is Always a Subspace

**Theorem (Larson, Theorem 4.6).** If $S = \{\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k\}$
is a set of vectors in a vector space $V$, then $\text{span}(S)$ is a subspace of $V$.

**Proof sketch:**
1. **Zero vector:** $\mathbf{0} = 0\mathbf{v}_1 + 0\mathbf{v}_2 + \cdots + 0\mathbf{v}_k \in \text{span}(S)$. \checkmark
2. **Closed under addition:** If $\mathbf{u} = \sum a_i \mathbf{v}_i$ and
   $\mathbf{w} = \sum b_i \mathbf{v}_i$, then
   $\mathbf{u} + \mathbf{w} = \sum (a_i + b_i)\mathbf{v}_i \in \text{span}(S)$. \checkmark
3. **Closed under scalar multiplication:** If $\mathbf{u} = \sum a_i \mathbf{v}_i$, then
   $c\mathbf{u} = \sum (ca_i)\mathbf{v}_i \in \text{span}(S)$. \checkmark

This means **every span is a subspace**. In fact, the converse direction is also true
in finite-dimensional spaces: every subspace *is* the span of some set of vectors
(its basis). We will make this precise in Section 4.5.

Let's demonstrate with `is_in_span` from our library.

In [ ]:
# S = {(1, 0, 1), (0, 1, 1)} spans a plane in R^3.
S = [[1, 0, 1], [0, 1, 1]]

# Test: is (3, 2, 5) in span(S)?  Should be: 3*(1,0,1) + 2*(0,1,1) = (3,2,5)
v1 = [3, 2, 5]
in_span, coeffs = is_in_span(v1, S)
print(f"v = {v1}")
print(f"In span(S)? {in_span}")
print(f"Coefficients: {coeffs}")
if coeffs:
    recon = [sum(coeffs[j] * S[j][i] for j in range(len(S))) for i in range(len(v1))]
    print(f"Reconstruction: {recon}")

print()

# Test: is (1, 1, 1) in span(S)?  Need a + 0 = 1, 0 + b = 1, a + b = 1 => a=b=... no contradiction
v2 = [1, 1, 1]
in_span, coeffs = is_in_span(v2, S)
print(f"v = {v2}")
print(f"In span(S)? {in_span}")
if coeffs:
    print(f"Coefficients: {coeffs}")

print()

# Test: is (1, 0, 0) in span(S)?  Need a = 1, b = 0, a + b = 0 => 1 = 0. Contradiction!
v3 = [1, 0, 0]
in_span, coeffs = is_in_span(v3, S)
print(f"v = {v3}")
print(f"In span(S)? {in_span}")
print("(1,0,0) is NOT in the plane spanned by S — it sticks out of that plane.")

print()

# Verify subspace conditions on span(S) by sampling
print("--- Subspace verification on span(S) ---")

def in_span_S(v):
    result, _ = is_in_span(v, S)
    return result

zero_3 = [0.0, 0.0, 0.0]
span_samples = [[1, 0, 1], [0, 1, 1], [3, 2, 5], [-1, 4, 3], [1, 1, 2]]
ok, report = is_subspace_candidate(span_samples, zero_3, in_span_S)
print(f"span(S) passes subspace test: {ok}")

### 3.6 Null Space Preview — Solutions of $A\mathbf{x} = \mathbf{0}$

The **null space** (or **kernel**) of an $m \times n$ matrix $A$ is:

$$\text{Null}(A) = \{\mathbf{x} \in \mathbb{R}^n : A\mathbf{x} = \mathbf{0}\}$$

**Theorem.** $\text{Null}(A)$ is a subspace of $\mathbb{R}^n$.

**Proof using the three conditions:**
1. **Contains $\mathbf{0}$:** $A\mathbf{0} = \mathbf{0}$, so $\mathbf{0} \in \text{Null}(A)$. \checkmark
2. **Closed under addition:** If $A\mathbf{u} = \mathbf{0}$ and $A\mathbf{v} = \mathbf{0}$, then
   $A(\mathbf{u} + \mathbf{v}) = A\mathbf{u} + A\mathbf{v} = \mathbf{0} + \mathbf{0} = \mathbf{0}$. \checkmark
3. **Closed under scalar multiplication:** If $A\mathbf{u} = \mathbf{0}$, then
   $A(c\mathbf{u}) = cA\mathbf{u} = c\mathbf{0} = \mathbf{0}$. \checkmark

Let's verify this computationally.

In [ ]:
# A = [[1, 2, 1], [2, 4, 2]]  — rank 1, so null space is 2-dimensional
A = Matrix.from_lists([[1, 2, 1], [2, 4, 2]])
print("A ="); print(A)

# Solve Ax = 0  (augmented matrix [A | 0])
aug = Matrix(A.rows, A.cols + 1)
for i in range(A.rows):
    for j in range(A.cols):
        aug[i, j] = A[i, j]

sol_type, result = solve(aug)
print(f"\nSolution type: {sol_type}")
print(f"Particular solution: {result['particular']}")
print(f"Free variables: {result['free_vars']}")
print(f"Direction vectors:")
for fv, direction in result['directions'].items():
    print(f"  x{fv+1} free -> direction = {direction}")

print()

# Extract null space basis vectors
ns_basis = [result['directions'][fv] for fv in result['free_vars']]
print(f"Null space basis: {ns_basis}")

print()
print("--- Verify the 3 subspace conditions ---")

# Condition 1: zero vector is in null space
zero_vec = [0.0, 0.0, 0.0]
Az = multiply(A, Matrix.from_vector(zero_vec))
is_zero = all(abs(Az[i, 0]) < EPSILON for i in range(Az.rows))
print(f"1. A * 0 = 0? {is_zero}")

# Condition 2: closed under addition
u = ns_basis[0]
v = ns_basis[1]
w = [u[i] + v[i] for i in range(len(u))]
Aw = multiply(A, Matrix.from_vector(w))
is_zero_sum = all(abs(Aw[i, 0]) < EPSILON for i in range(Aw.rows))
print(f"2. u, v in Null(A) => u+v in Null(A)? {is_zero_sum}")
print(f"   u = {u}, v = {v}, u+v = {w}")

# Condition 3: closed under scalar multiplication
c = 7.0
cu = [c * u[i] for i in range(len(u))]
Acu = multiply(A, Matrix.from_vector(cu))
is_zero_scaled = all(abs(Acu[i, 0]) < EPSILON for i in range(Acu.rows))
print(f"3. u in Null(A) => {c}*u in Null(A)? {is_zero_scaled}")
print(f"   {c}*u = {cu}")

print()
print("All 3 conditions satisfied: Null(A) is a subspace.")

# Bonus: test with a random linear combination
a, b = 3.5, -2.0
combo = [a * ns_basis[0][i] + b * ns_basis[1][i] for i in range(3)]
A_combo = multiply(A, Matrix.from_vector(combo))
is_zero_combo = all(abs(A_combo[i, 0]) < EPSILON for i in range(A_combo.rows))
print(f"\nBonus: {a}*n1 + {b}*n2 = {combo}")
print(f"A * (linear combo) = 0? {is_zero_combo}")

---

## 4. Visualize

### 4.1 $\mathbb{R}^2$: Subspace vs. Non-Subspace

We plot the line $y = 2x$ (a subspace) alongside the shifted line $y = 2x + 1$
(not a subspace). The subspace passes through the origin; the shifted line does not.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t = np.linspace(-3, 3, 300)

# --- Left panel: y = 2x (subspace) ---
ax = axes[0]
ax.plot(t, 2 * t, 'b-', linewidth=2, label=r'$y = 2x$')
ax.plot(0, 0, 'ko', markersize=8, zorder=5)
ax.annotate('origin', (0, 0), textcoords='offset points', xytext=(10, -15),
            fontsize=10, color='black')

# Show closure: v and 2v both on the line
ax.plot(1, 2, 'ro', markersize=8, zorder=5)
ax.annotate(r'$\mathbf{v} = (1,2)$', (1, 2), textcoords='offset points',
            xytext=(10, 5), fontsize=10, color='red')
ax.plot(2, 4, 'rs', markersize=8, zorder=5)
ax.annotate(r'$2\mathbf{v} = (2,4)$', (2, 4), textcoords='offset points',
            xytext=(10, 5), fontsize=10, color='red')

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-6.5, 6.5)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(r'Subspace: $y = 2x$', fontsize=13)
ax.legend(fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Right panel: y = 2x + 1 (NOT a subspace) ---
ax = axes[1]
ax.plot(t, 2 * t + 1, 'r-', linewidth=2, label=r'$y = 2x + 1$')
ax.plot(0, 0, 'ko', markersize=8, zorder=5)
ax.annotate('origin', (0, 0), textcoords='offset points', xytext=(10, -15),
            fontsize=10, color='black')

# Show failure: (0,1) and (1,3) are on the line, but (0,1)+(1,3) = (1,4) is NOT
ax.plot(0, 1, 'go', markersize=8, zorder=5)
ax.annotate(r'$(0,1)$', (0, 1), textcoords='offset points',
            xytext=(-35, 5), fontsize=10, color='green')
ax.plot(1, 3, 'go', markersize=8, zorder=5)
ax.annotate(r'$(1,3)$', (1, 3), textcoords='offset points',
            xytext=(10, 5), fontsize=10, color='green')
ax.plot(1, 4, 'gx', markersize=12, markeredgewidth=3, zorder=5)
ax.annotate(r'$(1,4)$ NOT on line', (1, 4), textcoords='offset points',
            xytext=(10, 5), fontsize=10, color='green')

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-6.5, 6.5)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(r'NOT a subspace: $y = 2x + 1$', fontsize=13)
ax.legend(fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 $\mathbb{R}^3$: Plane Through the Origin

We visualize the plane $x + y + z = 0$ in $\mathbb{R}^3$. This plane is spanned by
the vectors $(1, -1, 0)$ and $(0, 1, -1)$. As a subspace, any linear combination
of these two vectors stays within the plane.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plane x + y + z = 0  =>  z = -x - y
xx, yy = np.meshgrid(np.linspace(-2, 2, 20), np.linspace(-2, 2, 20))
zz = -xx - yy

ax.plot_surface(xx, yy, zz, alpha=0.25, color='steelblue', edgecolor='none')

# Basis vectors of the plane
origin = [0, 0, 0]
v1 = [1, -1, 0]
v2 = [0, 1, -1]

ax.quiver(*origin, *v1, color='red', arrow_length_ratio=0.1, linewidth=2.5,
          label=r'$\mathbf{v}_1 = (1,-1,0)$')
ax.quiver(*origin, *v2, color='darkgreen', arrow_length_ratio=0.1, linewidth=2.5,
          label=r'$\mathbf{v}_2 = (0,1,-1)$')

# A linear combination: 1.5*v1 + (-0.5)*v2
combo = [1.5 * v1[i] + (-0.5) * v2[i] for i in range(3)]
ax.quiver(*origin, *combo, color='purple', arrow_length_ratio=0.1, linewidth=2.5,
          label=r'$1.5\mathbf{v}_1 - 0.5\mathbf{v}_2$')

# Scatter some random points in the plane
np.random.seed(43)
for _ in range(30):
    a, b = np.random.uniform(-2, 2, 2)
    pt = [a * v1[i] + b * v2[i] for i in range(3)]
    ax.scatter(*pt, c='steelblue', s=15, alpha=0.6)

ax.scatter(0, 0, 0, c='black', s=60, zorder=5)
ax.text(0.1, 0.1, 0.1, 'origin', fontsize=10)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title(r'Subspace of $\mathbb{R}^3$: plane $x + y + z = 0$', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.view_init(elev=25, azim=135)

plt.tight_layout()
plt.show()

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Is the set $W = \{(x, y, z) \in \mathbb{R}^3 : x + y + z = 0\}$ a subspace of $\mathbb{R}^3$?

Verify all three conditions:
1. Is $\mathbf{0} = (0, 0, 0)$ in $W$?
2. Pick two vectors in $W$ and check that their sum is also in $W$.
3. Pick a vector in $W$ and a scalar, and check that the scalar multiple is in $W$.

Use the `is_subspace_candidate` function with an appropriate membership test.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Let $S = \{(1, 2, 0), (0, 1, -1), (1, 0, 2)\}$.

Determine whether $\mathbf{v} = (3, 5, -1)$ is in $\text{span}(S)$.

If yes, find the coefficients $c_1, c_2, c_3$ such that
$\mathbf{v} = c_1 \mathbf{s}_1 + c_2 \mathbf{s}_2 + c_3 \mathbf{s}_3$.

Use `is_in_span` and verify by reconstructing $\mathbf{v}$ from the coefficients.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `span_dimension(vectors)` that computes the dimension of $\text{span}(S)$
for a set of vectors $S$.

**Strategy:** Stack the vectors as columns of a matrix, reduce to RREF, and count the
number of pivot columns. The number of pivots equals $\dim(\text{span}(S))$.

Test your function on:
1. $S_1 = \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}$ — should give dimension 3.
2. $S_2 = \{(1, 2, 3), (2, 4, 6)\}$ — should give dimension 1 (second vector is a multiple).
3. $S_3 = \{(1, 0, 1), (0, 1, 1), (1, 1, 2)\}$ — should give dimension 2 (third = first + second).
4. $S_4 = \{(1, 0), (0, 1), (1, 1)\}$ — should give dimension 2 (three vectors in $\mathbb{R}^2$).

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Idea |
|---|---|
| **Subspace** | A nonempty subset of $V$ that is itself a vector space under the same operations |
| **Subspace test** | Check 3 conditions: contains $\mathbf{0}$, closed under $+$, closed under scalar $\cdot$ |
| **Trivial subspaces** | $\{\mathbf{0}\}$ and $V$ itself |
| **Subspaces of $\mathbb{R}^2$** | $\{\mathbf{0}\}$, lines through origin, $\mathbb{R}^2$ |
| **Subspaces of $\mathbb{R}^3$** | $\{\mathbf{0}\}$, lines through origin, planes through origin, $\mathbb{R}^3$ |
| **Span** | $\text{span}(S) = $ all linear combinations of $S$; always a subspace |
| **Null space** | $\text{Null}(A) = \{\mathbf{x} : A\mathbf{x} = \mathbf{0}\}$; always a subspace |

**Takeaway:** The subspace concept is the bridge between "set of vectors" and "vector space."
The three-condition test gives us a fast way to verify whether a set has the right structure.
Every span is a subspace, and every null space is a subspace — these two constructions will
generate all the subspaces we need for the rest of the course.